In [1]:
import pandas as pd
clean_df = pd.read_csv("clean_transactions.csv")

C:\Users\bunty\AppData\Local\Temp\ipykernel_21204\1488705549.py:2: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  clean_df = pd.read_csv("clean_transactions.csv")


In [2]:
clean_df.head()

,Customer ID,Invoice,InvoiceDate,Quantity,Price,total_amount
0,13085.0,489434,2009-12-01 07:45:00,12,6.95,83.4
1,13085.0,489434,2009-12-01 07:45:00,12,6.75,81.0
2,13085.0,489434,2009-12-01 07:45:00,12,6.75,81.0
3,13085.0,489434,2009-12-01 07:45:00,48,2.10,100.8
4,13085.0,489434,2009-12-01 07:45:00,24,1.25,30.0


In [3]:
clean_df["InvoiceDate"] = pd.to_datetime(clean_df["InvoiceDate"])


In [4]:
analysis_date = clean_df["InvoiceDate"].max()
analysis_date


Timestamp('2011-12-09 12:50:00')

In [5]:
recency_df = (
    clean_df
    .groupby("Customer ID")["InvoiceDate"]
    .max()
    .reset_index()
)

recency_df["recency"] = (analysis_date - recency_df["InvoiceDate"]).dt.days


In [7]:
recency_df.head()


,Customer ID,InvoiceDate,recency
0,12346.0,2011-01-18 10:01:00,325
1,12347.0,2011-12-07 15:52:00,1
2,12348.0,2011-09-25 13:13:00,74
3,12349.0,2011-11-21 09:51:00,18
4,12350.0,2011-02-02 16:01:00,309


In [8]:
frequency_df = (
    clean_df
    .groupby("Customer ID")["Invoice"]
    .nunique()
    .reset_index()
)

frequency_df.rename(columns={"Invoice": "frequency"}, inplace=True)


In [9]:
frequency_df.head()


,Customer ID,frequency
0,12346.0,12
1,12347.0,8
2,12348.0,5
3,12349.0,4
4,12350.0,1


In [10]:
monetary_df = (
    clean_df
    .groupby("Customer ID")["total_amount"]
    .sum()
    .reset_index()
)

monetary_df.rename(columns={"total_amount": "monetary"}, inplace=True)


In [11]:
monetary_df.head()


,Customer ID,monetary
0,12346.0,77556.46
1,12347.0,5633.32
2,12348.0,2019.40
3,12349.0,4428.69
4,12350.0,334.40


In [12]:
rfm_df = recency_df.merge(frequency_df, on="Customer ID")
rfm_df = rfm_df.merge(monetary_df, on="Customer ID")


In [13]:
rfm_df.head()


,Customer ID,InvoiceDate,recency,frequency,monetary
0,12346.0,2011-01-18 10:01:00,325,12,77556.46
1,12347.0,2011-12-07 15:52:00,1,8,5633.32
2,12348.0,2011-09-25 13:13:00,74,5,2019.40
3,12349.0,2011-11-21 09:51:00,18,4,4428.69
4,12350.0,2011-02-02 16:01:00,309,1,334.40


In [14]:
rfm_df = rfm_df.drop(columns=["InvoiceDate"])


In [15]:
rfm_df.head()


,Customer ID,recency,frequency,monetary
0,12346.0,325,12,77556.46
1,12347.0,1,8,5633.32
2,12348.0,74,5,2019.40
3,12349.0,18,4,4428.69
4,12350.0,309,1,334.40


In [16]:
rfm_df.describe()


,Customer ID,recency,frequency,monetary
count,5878.000000,5878.000000,5878.000000,5878.000000
mean,15315.313542,200.331916,6.289554,3018.616737
std,1715.572666,209.338707,13.009788,14737.731040
min,12346.000000,0.000000,1.000000,2.950000
25%,13833.250000,25.000000,1.000000,348.762500
50%,15314.500000,95.000000,3.000000,898.915000
75%,16797.750000,379.000000,7.000000,2307.090000
max,18287.000000,738.000000,398.000000,608821.650000


In [17]:
rfm_df.isna().sum()


Customer ID    0
recency        0
frequency      0
monetary       0
dtype: int64

In [19]:
rfm_df.to_csv(
    "customer_rfm_features.csv",
    index=False
)
